<a href="https://colab.research.google.com/github/aledamelio/ML_Pavia/blob/main/08_Nonparametric_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Nonlinear Classification with Support Vector Machines (SVM)

First, let's review *linear* (soft-margin) SVMs.

We try to classify the **moons dataset** using SVM with a linear kernel

In [ ]:
# VISUALIZATION

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

def plot_decision_regions(X, y, classifier, resolution=0.02,
                          support_vectors=None):
    X, y = np.asarray(X), np.asarray(y)
    classes = np.unique(y)
    if len(classes) != 2:
        raise ValueError("This function requires binary classification.")

    colors = ("red", "blue")
    markers = ("s", "x")
    cmap = ListedColormap(colors)

    # Grid covering the data
    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx1, xx2 = np.meshgrid(
        np.arange(x1_min, x1_max, resolution),
        np.arange(x2_min, x2_max, resolution)
    )
    grid = np.c_[xx1.ravel(), xx2.ravel()]

    # Colored class regions
    predictions = classifier.predict(grid)
    Z = np.searchsorted(classes, predictions).reshape(xx1.shape)
    plt.contourf(xx1, xx2, Z, levels=[-0.5, 0.5, 1.5],
                 alpha=0.2, cmap=cmap)

    # Decision boundary and margins: f(x) = 0 and f(x) = ±1
    scores = classifier.decision_function(grid).reshape(xx1.shape)
    plt.contour(
        xx1, xx2, scores,
        levels=[-1, 0, 1],
        colors="black",
        linestyles=["--", "-", "--"],
        linewidths=[1, 1.5, 1]
    )

    # Samples
    for idx, cl in enumerate(classes):
        edge_kwargs = {} if markers[idx] == "x" else {"edgecolor": "black"}
        plt.scatter(
            X[y == cl, 0], X[y == cl, 1],
            color=colors[idx], marker=markers[idx],
            alpha=0.8, label=cl, zorder=3, **edge_kwargs
        )

    # Highlight supplied samples with rings
    if support_vectors is not None:
        sv = np.asarray(support_vectors).reshape(-1, 2)
        if len(sv):
            plt.scatter(
                sv[:, 0], sv[:, 1],
                s=160, facecolors="none", edgecolors="black",
                linewidths=1.5, label="Support vectors", zorder=4
            )

    plt.xlim(x1_min, x1_max)
    plt.ylim(x2_min, x2_max)
    plt.gca().set_aspect("equal", adjustable="box")

In [ ]:
# SYNTHETIC DATASET GENERATION

from sklearn.datasets import make_moons

X, y = make_moons(noise=0.3, random_state=0)

# plot data
cm_bright = ListedColormap(['red', 'blue'])
plt.scatter(X[:, 0], X[:, 1], c=y, cmap=cm_bright, edgecolors='k')

plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

plt.show()

In [ ]:
from sklearn import linear_model
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# We use a pipeline but give a name to the individual steps
linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="linear", C=1.0)),
])

linear_model.fit(X, y)

# We use the given names to access individual steps in the pipeline
scaler = linear_model.named_steps["scaler"]
svm = linear_model.named_steps["svm"]

# In this way, it is easy to extract the support vectors
sv = svm.support_vectors_
sv_indices = svm.support_

plot_decision_regions(
    X, y, classifier=linear_model,
    support_vectors=sv  # Do you notice anything strange?
)
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend(loc="upper left")
plt.title(f"Linear SVM")

plt.tight_layout()
plt.show()

To obtain a nonlinear decision boundary (and margin), we can apply **polynomial features** to our data

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly_model = Pipeline([
    ("poly", PolynomialFeatures(degree=3, include_bias=False)), # SVM is already learning an intercept; try changing the degree
    ("scaler", StandardScaler()),  # Scale after features (scaling before may help with very large degrees)
    ("svm", SVC(kernel="linear", C=1.0)),
])

poly_model.fit(X, y)

scaler = poly_model.named_steps["scaler"]
svm = poly_model.named_steps["svm"]

sv = svm.support_vectors_
sv_indices = svm.support_

plot_decision_regions(
    X, y, classifier=poly_model,
    support_vectors=X[sv_indices]  # localize support vectors in input space
)
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend(loc="upper left")
plt.title(f"SVM on polynomial features")

plt.tight_layout()
plt.show()

Using explicit polynomial features is unnecessary: a **polynomial kernel** does the same job (with small degree and/or large datasets, consider using PolynomialFeatures with **LinearSVC** for greater efficiency)

Do you remember the common kernels?

__Linear Kernel:__ $k(\mathbf{x},\mathbf{z}) = \mathbf{x}^\top\mathbf{z}$

__Polynomial Kernel:__ $k(\mathbf{x},\mathbf{z}) = (\gamma\mathbf{x}^\top\mathbf{z}+r)^d$

Gaussian or __RBF Kernel:__ $k(\mathbf{x},\mathbf{z})=\exp(-\gamma \|\mathbf{x}-\mathbf{z}\|^2)$

### Soft-margin Kernel SVM: Dual Program

$$
\begin{aligned}
\max_{\boldsymbol\alpha}\quad&\sum_{n=1}^N\alpha_n
-\frac12\sum_{n=1}^N\sum_{m=1}^N\alpha_n\alpha_mt_nt_mk(\mathbf{x}_n,\mathbf{x}_m)\\
\text{subject to}\quad&0\leq\alpha_n\leq C,\quad n=1,\ldots,N,\\
&\sum_{n=1}^N\alpha_nt_n=0
\end{aligned}
$$

Samples $\mathbf{x}_n$ with $\alpha_n>0$ are __support vectors__. Those with $\alpha_n=C$ correspond to __margin violations__.

__Prediction__ based on support vectors: $\widehat{t}(\mathbf{x})=\mathrm{sign}\left(f(\mathbf{x})\right)$ where the *score* is
$$
  f(\mathbf{x}) = \sum_{n:\alpha_n>0}\alpha_nt_nk(\mathbf{x}_n,\mathbf{x})+w_0
$$

The intercept is computed as
$w_0=t_s-\sum_{n:\alpha_n>0}\alpha_nt_nk(\mathbf{x}_n,\mathbf{x}_s)$ given any support vector $\mathbf{x}_s$ with $0<\alpha_s<C$.

The decision boundary (in input space) is the set of $\mathbf{x}$ such that $f(\mathbf{x})=0$. The margin is delimited by $f(\mathbf{x})=\pm 1$ (corresponding to margin planes in feature space).

Check the [docs](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) on how to use kernels within SVC

In [ ]:
poly_kernel_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="poly",
        degree=3,
        coef0=1.0,      # Include lower-degree terms (default 0 gives only degree 3)
        gamma="scale",  # This is default. It uses a data-dependent coefficient
        C=1.0,
    )),
])

poly_kernel_model.fit(X, y)

svm = poly_kernel_model.named_steps["svm"]

sv_indices = svm.support_

plot_decision_regions(
    X, y, classifier=poly_kernel_model,
    support_vectors=X[sv_indices]  # Locate support vectors in original input space
)
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend(loc="upper left")
plt.title(f"SVM with polynomial kernel (degree={svm.degree})")

plt.tight_layout()
plt.show()

### Pros and Cons on nonparametric methods: a practical perspective
__Pros:__
* Flexibility: can capture complex patterns without specifying a fixed finite-dimensional model

* Adaptability to the data: model complexity can grow with the training set

__Cons:__
* Memory costs: often retain many training samples for prediction

* Costly prediction: prediction relies on retained samples and can be slow, especially with large training datasets

* Interpretability: predictions can be harder to explain than those of simple parametric models

### RBF Kernel SVM

Let's try an __RBF Kernel__ next. Now the "gamma" parameter corresponds to $1/(2\sigma^2)$ where $\sigma$ is the **bandwitdh**
* Larger gamma (smaller $\sigma$): boundary is more sensitive to local structure
* Smaller gamma (larger $\sigma$): smoother boundary

In [ ]:
rbf_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf", # rbf is the default
        gamma=0.5,  # Default: data-dependent coefficient; try different numerical values
        C=1.0,
    )),
])

rbf_model.fit(X, y)

svm = rbf_model.named_steps["svm"]

sv_indices = svm.support_

plot_decision_regions(
    X, y, classifier=rbf_model,
    support_vectors=X[sv_indices]
)
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend(loc="upper left")
plt.title("SVM with RBF kernel")

plt.tight_layout()
plt.show()

There are two important hyperparameters here: the penalization coefficient $C$ and the inverse bandwidth $\gamma$. In both cases a larger value leads to a more intricate decision boundary and a higher sensitivity to the training data, although the reason is different:
* Larger $\gamma$ (smaller $\sigma$) amplifies the effect support vector have on classifying nearby points
* Larger $C$ penalizes margin violations more and forces the boundary to separate the classes in the training set

In both cases the model will tend to have smaller bias and larger variance. However, the two two hyperparameters interact in ways that are hard to predict. We will select them both using a grid search in cross validation, using scikit-learn's **GridSearchCV**

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Generate a larger moons dataset
X, y = make_moons(n_samples=200, noise=0.25, random_state=42)

# 80–20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

rbf_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf")),
])

# Select C and gamma using grid search and 5-fold CV
param_grid = {
    "svm__C": np.logspace(-2, 3, 6),      # 0.01, ..., 1000
    "svm__gamma": np.logspace(-3, 2, 6),  # 0.001, ..., 100
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(
    estimator=rbf_model,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    refit=True,  # Automatically retrain the best model on all training data
)
search.fit(X_train, y_train)

# 4. Evaluate the retrained best model on the held-out test set
best_rbf_model = search.best_estimator_
y_pred = best_rbf_model.predict(X_test)

print(f"Best C: {search.best_params_['svm__C']:g}")
print(f"Best gamma: {search.best_params_['svm__gamma']:g}")
print(f"Best mean CV accuracy: {search.best_score_:.3f}")
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.3f}")

In [ ]:
# Training data with support vectors
plot_decision_regions(
    X_train, y_train, classifier=best_rbf_model,
    support_vectors=X_train[best_rbf_model.named_steps["svm"].support_]
)
plt.show()


In [ ]:
# Test data
plot_decision_regions(
    X_test, y_test, classifier=best_rbf_model,
    support_vectors=None
)
plt.show()

### Gaussian Kernel is not the same as Gaussian features

We can do linear classification with a finite number of Gaussian features of the form

$\phi_j(\mathbf{x}) = \exp(-\gamma\|\mathbf{x}-\mathbf{c}_j\|^2)$

But the centers $c_j$ are fixed and we have to manually select them.

The Gaussian (RBF) kernel **adapts** to the data: every support vector acts as a center, and there as many support vector as are needed to define the margin, not more!

In [ ]:
from sklearn.preprocessing import FunctionTransformer

# Nine predefined centers, independent of the training samples
centers = np.array([
    [cx, cy]
    for cx in np.arange(-1.0, 2.0, 1.) # try to increase the number of centers
    for cy in np.arange(-0.5, 1.5, 1.)
])

def gaussian_features(X, centers, gamma):
    return np.column_stack([
        np.exp(-gamma * np.sum((X - center)**2, axis=1))
        for center in centers
    ])

gaussian_model = Pipeline([
    ("gaussian", FunctionTransformer(
        gaussian_features,
        kw_args={"centers": centers, "gamma": 1.0}, # Try changing gamma
    )),
    ("scaler", StandardScaler()),  # Scale feature outputs -> centers stay fixed
    ("svm", SVC(kernel="linear", C=1.0)),
])

gaussian_model.fit(X_train, y_train)

print(f"Test accuracy: {gaussian_model.score(X_test, y_test):.3f}")

plt.figure(figsize=(7, 5))
plot_decision_regions(
    X_test, y_test, classifier=gaussian_model
)

# Show the fixed Gaussian centers in the original input space
plt.scatter(
    centers[:, 0], centers[:, 1],
    marker="+", c="black", s=180, linewidths=2,
    label="Fixed Gaussian centers", zorder=5,
)

plt.xlabel("x1")
plt.ylabel("x2")
plt.legend(loc="upper left")
plt.title(f"Linear SVM with {len(centers)} fixed Gaussian features")
plt.tight_layout()
plt.show()

### Custom Kernels

You may want to design your own kernel for your specific application. Remind that a kernel function $k(\mathbf{x}, \mathbf{z})$:
* Should be a measure of similarity
* Must be symmetric: $k(\mathbf{x}, \mathbf{z})=k(\mathbf{z}, \mathbf{x})$
* Must be positive semi-definite: the Gram matrix $K$ must be p.s.d. (all eigenvalues must be non-negative) for any dataset size:
$$
  \mathbf{K}=\begin{bmatrix}
k(\mathbf{x}_1,\mathbf{x}_1)&\cdots&k(\mathbf{x}_1,\mathbf{x}_N)\\
\vdots&\ddots&\vdots\\
k(\mathbf{x}_N,\mathbf{x}_1)&\cdots&k(\mathbf{x}_N,\mathbf{x}_N)
\end{bmatrix}
$$

A way to construct valid kernels is to start from a known one (such as the linear kernel) and apply __operations that preserve the validity of the kernel:__
* Generalized linear kernel: if $A\in\mathbb{R}^{d\times d}$ is a symmetric positive semi-definite matrix, then $k(\mathbf{x}, \mathbf{z}) = \mathbf{x}^\top A \mathbf{z}$ is a valid kernel
* Feature map: if $k(\mathbf{x},\mathbf{z})$ is a valid kernel and $\phi:\mathbb{R}^d\to\mathbb{R}^m$, also $k(\phi(\mathbf{x}),\phi(\mathbf{z}))$ is a valid kernel
* Sum: if $k_1$ and $k_2$ are valid kernels so is $k(\mathbf{x}, \mathbf{z})=k_1(\mathbf{x}, \mathbf{z})+k_2(\mathbf{x}, \mathbf{z})$
* Multiplication by a non-negative scalar: if $k$ is a valid kernel and $a\ge 0$, then $k'(\mathbf{x}, \mathbf{z}) = ak(\mathbf{x}, \mathbf{z})$ is a valid kernel
* Offset: if $k$ is a valid kernel and $c\ge 0$, then $k'(\mathbf{x}, \mathbf{z})=k(\mathbf{x}, \mathbf{z})+c$ is a valid kernel
* Pointwise product: if $k_1$ and $k_2$ are valid kernels so is $k(\mathbf{x}, \mathbf{z})=k_1(\mathbf{x}, \mathbf{z})k_2(\mathbf{x}, \mathbf{z})$
* Integer powers: if $k$ is a valid kernel and $p\in\{1,2,\dots\}$, $k_p(\mathbf{x},\mathbf{z})=k(\mathbf{x},\mathbf{z})^p$ is a valid kernel
* Exponential: if $k$ is a valid kernel, then $k'(\mathbf{x}, \mathbf{z}) = \exp(k(\mathbf{x}, \mathbf{z}))$ is a valid kernel
* Rescaling: if $k$ is a valid kernel and $f:\mathbb{R}^d\to\mathbb{R}$, also $k'(\mathbf{x},\mathbf{z}) = f(\mathbf{x})k(\mathbf{x},\mathbf{z})f(\mathbf{z})$ is a valid kernel
* Normalization: if $k$ is a valid kernel and $k(\mathbf{x},\mathbf{x})>0$ for all $\mathbf{x}$, then also $k'(\mathbf{x},\mathbf{z}) = \frac{k(\mathbf{x},\mathbf{z})}{\sqrt{k(\mathbf{x},\mathbf{x})k(\mathbf{z},\mathbf{z})}}$ is a valid kernel

These operations can be arbitrarily combined: for example, $k(\mathbf{x},\mathbf{z}) = 2\mathbf{x}^\top\mathbf{z} + \exp(\mathbf{x}^\top\mathbf{z}+1)$ is a valid kernel

__Exercise:__
1. Use the code below to try different valid custom kernels on the Moons dataset
2. What happens if you pass a non-valid kernel to SVC?
3. Can you derive the RBF kernel from the linear kernel using any number of these operations?

There is a catch: SVC wants a kernel function that acts on *input matrices* of size $N\times D$ and computes the whole Gram matrix (that is, all pairwise kernel evaluations)

In [ ]:
from sklearn.metrics.pairwise import pairwise_kernels  # method to compute Gram matrices

# Linear kernel you can customize.
def custom_kernel(x, z):
    return np.dot(x, z)

# Adapter for SVC, which passes two batches of inputs
def kernel(A, B):
    return pairwise_kernels(A, B, metric=custom_kernel)

custom_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel=kernel, C=1.0)),
])

custom_model.fit(X_train, y_train)

print(f"Test accuracy: {custom_model.score(X_test, y_test):.3f}")

sv_indices = custom_model.named_steps["svm"].support_

plt.figure(figsize=(7, 5))

# Use a large resolution: custom kernel computations take time
plot_decision_regions(
    X_train, y_train, classifier=custom_model,
    support_vectors=X_train[sv_indices],
    resolution=0.2
)
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend(loc="upper left")
plt.title("SVM with custom kernel")
plt.tight_layout()
plt.show()

Efficiency can be improved by pre-computing the Gram matrix

In [ ]:
from sklearn.metrics.pairwise import pairwise_kernels  # method to compute Gram matrices

# Linear kernel you can customize.
def custom_kernel(x, z):
    return np.dot(x, z)

# Fit the scaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Training–training and test–training Gram matrices
K_train = pairwise_kernels(
    X_train_scaled, metric=custom_kernel
)
K_test = pairwise_kernels(
    X_test_scaled, X_train_scaled, metric=custom_kernel
)

custom_model = SVC(kernel="precomputed", C=1.0)
custom_model.fit(K_train, y_train)  # Note we are passing the Gram matrix directly

print(f"Test accuracy: {custom_model.score(K_test, y_test):.3f}")

## Multi-class Classification with SVMs

SVC automatically handles multi-class classification ($m>2$ classes) using the **"one-versus-one"** strategy: it trains a binary SVM classifier for each pair of classes, for a total of
$$
  \frac{m(m-1)}{2}
$$
classifiers. At prediction, each binary classifier votes for its winning class and the class with more votes is the prediction.

In [ ]:
def plot_multi_class(X, y, classifier, resolution=0.02, feature_map=None):

    # setup marker generator and color map
    markers = ('s', 'x', 'o', '^', 'v')
    colors = ('red', 'blue', 'lightgreen', 'gray', 'cyan')
    cmap = ListedColormap(colors[:len(np.unique(y))])

    # plot the decision surface
    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx1, xx2 = np.meshgrid(np.arange(x1_min, x1_max, resolution),
                           np.arange(x2_min, x2_max, resolution))

    if feature_map == None:
        Z = classifier.predict(np.array([xx1.ravel(), xx2.ravel()]).T)
    else:
        Z = classifier.predict(feature_map.transform(np.array([xx1.ravel(), xx2.ravel()]).T))
    Z = Z.reshape(xx1.shape)
    plt.contourf(xx1, xx2, Z, alpha=0.3, cmap=cmap)
    plt.xlim(xx1.min(), xx1.max())
    plt.ylim(xx2.min(), xx2.max())

    # plot class examples
    for idx, cl in enumerate(np.unique(y)):
      plt.scatter(
          x=X[y == cl, 0],
          y=X[y == cl, 1],
          alpha=0.8,
          c=colors[idx],
          marker=markers[idx],
          label=cl,
          edgecolor='black' if markers[idx] != 'x' else None,
      )

In [ ]:
from sklearn import datasets
from sklearn.preprocessing import StandardScaler

iris = datasets.load_iris()
X = iris.data[:, [2, 3]]
y = iris.target

sc = StandardScaler()
X = sc.fit_transform(X)

svm = SVC(kernel='rbf', random_state=1, gamma=0.10, C=10.0)
svm.fit(X, y)

plot_multi_class(X, y,classifier=svm)
plt.legend(loc='upper left')
plt.tight_layout()
plt.title('RBF SVM for three classes')
plt.xlabel('petal length [standardized]')
plt.ylabel('petal width [standardized]')
plt.show()

# KNN Classifier

Predict the majority class among the $K$ Nearest Neighbors: the $K$ training inputs closest to $\mathbf{x}$

* Very simple
* Immediate extension to more than 2 classes
* No training
* __Non-parametric:__ model complexity depends on the number of training examples
* $K$ acts as a regularization hyperparameter: higher $K$, smoother boundary
* Memory cost: must retain *all* training examples
* Slow prediction: Needs to compare $\mathbf{x}$ with all training inputs. There are some algorithmic optimizations to make it faster: see the [docs](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)
* Works with different notions of distance: see some examples [here](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.distance_metrics.html#sklearn.metrics.pairwise.distance_metrics)

### KNN in scikit-learn

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn import datasets

iris = datasets.load_iris()
X = iris.data[:, [2, 3]]
y = iris.target

X = StandardScaler().fit_transform(X)  # Try to standardize the data
knn = KNeighborsClassifier(n_neighbors=5,  # Try to change the value of K
                           metric='euclidean',  # Try to change the metric
                           weights='uniform',  # Try 'distance' for distance-weighted KNN
                           algorithm='brute',  # Try 'ball_tree’ or ‘kd_tree’ for faster prediction
                           )
knn.fit(X, y)

plot_multi_class(X, y, classifier=knn)

plt.xlabel('petal length [standardized]')
plt.ylabel('petal width [standardized]')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

### Custom distances

A valid distance function of **metric** $d$ must have the following properties:
1. $d(\mathbf{x}, \mathbf{x})=0$ for all $\mathbf{x}$
2. $d(\mathbf{x}, \mathbf{z})>0$ for all $\mathbf{x}\neq \mathbf{z}$
3. Symmetry: $d(\mathbf{x}, \mathbf{z})=d(\mathbf{z}, \mathbf{x})$ for all $\mathbf{x},\mathbf{z}$
4. Triangle inequality: $d(\mathbf{x}, \mathbf{z})\le d(\mathbf{x}, \mathbf{y})+d(\mathbf{y}, \mathbf{z})$ for all $\mathbf{x},\mathbf{y},\mathbf{z}$

A **pseudo-metric** replaces the second property with $d(\mathbf{x}, \mathbf{z})\ge0$ for all $\mathbf{x}\neq \mathbf{z}$: two distinct points *can* have distance zero according to a pseudo-metric.
KNN still works with pseudo-metrics since it implements **tie-breaking rules** both for equally distant neighbors (by order of appereance) and equally represented classes (by lexicographical order)

* If $d$ is a metric and $a>0$, $d'(\mathbf{x}, \mathbf{z}) = a d(\mathbf{x}, \mathbf{z})$ is a metric
* If $d_1, d_2$ are metrics and $p\ge 1$, $d(\mathbf{x}, \mathbf{z})=(d_1(\mathbf{x}, \mathbf{z})^p+d_2(\mathbf{x}, \mathbf{z})^p)^{1/p}$ is a metric
* If $d_1, d_2$ are metrics, $d(\mathbf{x}, \mathbf{z})=\max\{d_1(\mathbf{x}, \mathbf{z}),d_2(\mathbf{x}, \mathbf{z})\}$ is a metric
* If $d$ is a metric and $0<\alpha\le 1$, $d'(\mathbf{x}, \mathbf{z}) = d(\mathbf{x}, \mathbf{z})^\alpha$ is a metric
* If $d$ is a metric, $d'(\mathbf{x}, \mathbf{z}) = \frac{d(\mathbf{x}, \mathbf{z})}{1+d(\mathbf{x}, \mathbf{z})}$ is a metric
* If $d$ is a metric, $d'(\mathbf{x}, \mathbf{z})=\log(1+d(\mathbf{x}, \mathbf{z}))$ is a metric
* If $d$ is a metric on $\mathbb{R}^M$ and $\phi:\mathbb{R}^D\to\mathbb{R}^M$ is a metric, $d'(\mathbf{x}, \mathbf{z})=d(\phi(\mathbf{x}), \phi(\mathbf{z}))$ is a pseudo-metric. If $\phi$ is injective, $d'$ is a metric.
* If $k$ is a valid kernel,
 $d_k(\mathbf{x},\mathbf{z}) = \sqrt{k(\mathbf{x},\mathbf{x})+k(\mathbf{z},\mathbf{z})-2k(\mathbf{x},\mathbf{z})}$ is a pseudo-metric

 __Exercise:__
1. Use the code below to try different distance functions on the Iris dataset
2. What happens if you pass a function that is not a (pseudo)metric?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Euclidean distance you can customize
def custom_distance(x, z):
    return np.sqrt(np.dot(x-z, x-z))

iris = load_iris()
X = iris.data[:, [2, 3]]
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

custom_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=5,
        metric=custom_distance,
        algorithm="ball_tree",  # kd_tree does not support custom distances
        weights="uniform",
    )),
])

custom_knn.fit(X_train, y_train)
print(f"Test accuracy: {custom_knn.score(X_test, y_test):.3f}")

# Use a large resolution: custom distance computation take time
plot_multi_class(X_test, y_test, classifier=custom_knn, resolution=0.1)

plt.xlabel('petal length [standardized]')
plt.ylabel('petal width [standardized]')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

Efficiency can be improved by pre-computing distances

In [ ]:
from sklearn.metrics import pairwise_distances  # this method takes care of computing all pairwise distances

# Euclidean distance you can customize.
def custom_distance(x, z):
    return np.sqrt(np.dot(x - z, x - z))

iris = load_iris()
X = iris.data[:, [2, 3]]
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Fit the scaler on training data only.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Training–training and test–training distance matrices
D_train = pairwise_distances(
    X_train_scaled, metric=custom_distance
)
D_test = pairwise_distances(
    X_test_scaled, X_train_scaled, metric=custom_distance
)

custom_knn = KNeighborsClassifier(
    n_neighbors=5,
    metric="precomputed",  # use precomputed distances
    weights="uniform",
)
custom_knn.fit(D_train, y_train)  # note we are passing the distances directly
print(f"Test accuracy: {custom_knn.score(D_test, y_test):.3f}")

### Exercise: Classification of DNA sequences

Each input is a DNA sequence: a string of 57 characters drawn from
the alphabet `A`, `C`, `G`, and `T`.

The target indicates whether the sequence is a **promoter**, a DNA region
involved in initiating gene transcription:

- `1`: promoter
- `0`: non-promoter

Unlike numerical measurements, the four nucleotides have no natural

**Your tasks:**

1. Either design a custom kernel function and train an SVM classifier, or design a custom distance function and train a KNN classifier
2. Report the accuracy on your classifier on the test set

Use the fixed, stratified train–test split provided below. Only use the test set to measure final accuracy.

**Hint:** <details>A typical distance between strings of the same length is the [*Hamming distance*](https://en.wikipedia.org/wiki/Hamming_distance): it just counts the number of positions at which the strings have different symbols.
If $d(s,t)$ is the Hamming distance of two strings of length $\ell$, $k(s,t) = \ell - d(s,t)$ is a valid kernel.</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Download the dataset: class label, sequence identifier, DNA sequence.
url = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "molecular-biology/promoter-gene-sequences/promoters.data"
)
data = pd.read_csv(
    url, header=None, names=["label", "identifier", "sequence"]
)

X = data["sequence"].str.strip().str.upper().to_numpy()
y = data["label"].str.strip().map({"-": 0, "+": 1}).to_numpy()

# Fixed 80–20 split, preserving the class proportions.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")
print(f"Sequence length:  {len(X_train[0])}")
print("\nExample training inputs (1 = promoter, 0 = non-promoter):")
for sequence, target in zip(X_train[:4], y_train[:4]):
    print(f"{sequence}  target={target}")